# EDA — Fraud_Data (E-commerce Transactions)

**Objective.** Understand the e-commerce transaction data, assess data quality, characterise the fraud signal, and quantify the class imbalance that will drive our resampling and evaluation choices.

All reusable logic lives in `src/` and is unit-tested; this notebook is the narrative layer on top.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable so `from src import ...` resolves.
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

from src import config
config.ensure_dirs()
FIG = config.FIGURES_DIR


In [ ]:
from src import data_loader as dl, cleaning
raw = dl.load_fraud_data()
print('Raw shape:', raw.shape)
raw.head()

## 1. Data overview & types

In [ ]:
raw.info()

In [ ]:
# signup_time / purchase_time arrive as strings; ip_address as float.
raw.dtypes

## 2. Data cleaning

Steps (each implemented and tested in `src/cleaning.py`):
1. **Correct dtypes** — parse timestamps to `datetime`, cast `source/browser/sex` to `category`.
2. **Duplicates** — drop exact duplicate rows.
3. **Missing values** — documented policy: drop columns >50% missing, median-impute numeric, mode-impute categorical.

In [ ]:
print('Missing values per column:')
display(cleaning.missing_value_report(raw) if not cleaning.missing_value_report(raw).empty else 'No missing values.')
print('Exact duplicate rows:', raw.duplicated().sum())

In [ ]:
df = cleaning.clean_fraud_data(raw)
print('Cleaned shape:', df.shape)
print('Shared devices (device_id used by >1 row):',
      (df.groupby('device_id').size() > 1).sum())
df.dtypes

**Note on duplicates / shared devices.** There are no exact duplicate rows, but thousands of `device_id`s are shared across transactions — we deliberately keep these because device-sharing is itself a fraud signal (engineered later as `device_user_count`).

## 3. Class imbalance

The headline constraint of the project: fraud is a small minority.

In [ ]:
from src.resampling import class_distribution
dist = class_distribution(df['class'])
display(dist)
fraud_rate = df['class'].mean()
print(f'Fraud rate: {fraud_rate:.2%}')

ax = sns.countplot(x='class', data=df)
ax.set(title=f'Class balance (fraud = {fraud_rate:.1%})',
       xlabel='class (0=legit, 1=fraud)', ylabel='count')
plt.savefig(FIG / 'fraud_class_balance.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Univariate distributions

Key numeric and categorical variables.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['purchase_value'], bins=40, ax=axes[0])
axes[0].set_title('Purchase value ($)')
sns.histplot(df['age'], bins=40, ax=axes[1])
axes[1].set_title('Age')
plt.savefig(FIG / 'fraud_univariate_numeric.png', dpi=120, bbox_inches='tight')
plt.show()
df[['purchase_value', 'age']].describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['source', 'browser', 'sex']):
    order = df[col].value_counts().index
    sns.countplot(x=col, data=df, order=order, ax=ax)
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=30)
plt.savefig(FIG / 'fraud_univariate_categorical.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Bivariate relationships with the target

How does each feature differ between fraudulent and legitimate transactions?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(x='class', y='purchase_value', data=df, ax=axes[0])
axes[0].set_title('Purchase value by class')
sns.boxplot(x='class', y='age', data=df, ax=axes[1])
axes[1].set_title('Age by class')
plt.savefig(FIG / 'fraud_bivariate_numeric.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Fraud rate by categorical level — the actionable view.
for col in ['source', 'browser', 'sex']:
    rate = df.groupby(col, observed=True)['class'].mean().sort_values(ascending=False)
    print(f'\nFraud rate by {col}:')
    print((rate * 100).round(2).astype(str) + '%')

## 6. Key findings

- **Imbalance:** ~9.4% of transactions are fraudulent — a minority class large enough for SMOTE but small enough that accuracy is a useless metric (a trivial all-legit classifier scores ~90.6%). We will evaluate with **AUC-PR, recall, precision and F1**.
- **Purchase value / age** distributions overlap substantially between classes — no single numeric feature cleanly separates fraud, motivating the engineered time/velocity features in `feature-engineering.ipynb`.
- **Channel/browser** fraud rates vary, providing categorical signal (one-hot encoded downstream).
- **Device sharing** is widespread and retained as a fraud signal.